# End-to-end detector / RT-DETRv2

Backbone, hybrid encoder, selected queries, reference boxes, decoder refinement, and query utilization. NMS-free but query count may limit dense-scene recall.

License and exact weight provenance are recorded in `LICENSES.md` and each run manifest. Approximate GPU requirements depend strongly on resolution, batch size, AMP, and the active Colab GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_USERNAME = "Harryphan72007"
GITHUB_REPOSITORY = "aerial-object-detection-benchmark"
DEFAULT_BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
REPO_DIR = f"/content/{GITHUB_REPOSITORY}"
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
assert GITHUB_USERNAME != "<MY_GITHUB_USERNAME>"
import os, sys, subprocess
if os.path.isdir(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, '.git')): raise RuntimeError(f'Existing non-Git directory: {REPO_DIR}')
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, '.git')): subprocess.run(['git', 'clone', '--branch', DEFAULT_BRANCH, REPO_URL, REPO_DIR], check=True)
from src.colab_setup import clone_or_update_repository, install_project, initialize_drive_directories, load_project_config, validate_drive_writable
clone_or_update_repository(REPO_URL, REPO_DIR, DEFAULT_BRANCH)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
install_project(REPO_DIR)
config = load_project_config('project_config.yaml')
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(DRIVE_ROOT)
print(subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip())
print(subprocess.run(['git', 'status', '--short'], capture_output=True, text=True, check=True).stdout)


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

## Editable experiment configuration

In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"
IMAGE_SIZE = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
NUM_EPOCHS = 100
SEED = 42
USE_AMP = True
RESUME_RUN_ID = None
RUN_HYPERPARAMETER_SEARCH = False
print(dict(MODEL_ID=MODEL_ID, DATASET_TRACK=DATASET_TRACK, IMAGE_SIZE=IMAGE_SIZE, EFFECTIVE_BATCH_SIZE=EFFECTIVE_BATCH_SIZE))

## Dataset validation

Validation checks image existence, dimensions, category IDs, bbox coordinates, zero-area boxes, and class coverage. Statistics expose class counts, size distributions, and objects per image.

In [ ]:
from src.data.validate_annotations import validate_coco
from src.data.statistics import compute_statistics
ann = paths.coco(DATASET_TRACK)/"annotations/instances_train.json"
report = validate_coco(ann, paths.coco(DATASET_TRACK)/"train")
print(report); report.raise_for_errors(); compute_statistics(ann)

## Model construction and introspection

The training command saves architecture, parameter totals, trainable/frozen totals, runtime config, and environment. After a first checkpoint, use notebook 08 for feature shapes, stage strides, FLOPs/MACs, and actual module names.

## Baseline training

Every epoch logs loss, LR, duration, gradient norm, memory, and validation metrics supported by the integration. `last.pth`, `best_map.pth`, and `best_aptiny.pth` are saved under the standardized run directory. Pass `RESUME_RUN_ID` after a Colab disconnect.

In [ ]:
cmd = f"python scripts/train.py --drive-root '{DRIVE_ROOT}' --model-id {MODEL_ID} --dataset-track {DATASET_TRACK} --image-size {IMAGE_SIZE} --batch-size {BATCH_SIZE} --gradient-accumulation-steps {GRADIENT_ACCUMULATION_STEPS} --epochs {NUM_EPOCHS} --seed {SEED}"
if not USE_AMP: cmd += " --no-amp"
if RESUME_RUN_ID: cmd += f" --resume-run-id {RESUME_RUN_ID}"
print(cmd)
!{cmd}

## RT-DETRv2 query/decoder visualization

Capture documented backbone/encoder/decoder modules and inspect predictions. Query-count and decoder-refinement analysis is expanded in notebook 08.


In [ ]:
from src.training.checkpointing import RunRegistry
from src.models.registry import create_adapter
from src.utils.serialization import read_yaml
from src.evaluation.visualization import select_module_names, capture_module_outputs, plot_activation_views, draw_predictions
from src.data.dataloaders import CocoDetectionRecords
from IPython.display import display
registry = RunRegistry(paths)
completed = registry.list_available_runs(MODEL_ID, DATASET_TRACK)
if completed:
    run = completed[0]
    run_dir = paths.run_dir(MODEL_ID, run["run_id"])
    model_cfg = read_yaml(run_dir / "model_config.yaml")
    model_cfg["input_resolution"] = run["input_resolution"]
    if run["framework"] in {"mmdetection", "vmamba_mmdetection"}:
        model_cfg["resolved_framework_config"] = str(run_dir / "runtime_config.py")
    adapter = create_adapter(MODEL_ID)
    model = adapter.load_model(registry.load_checkpoint_from_registry(run["run_id"]), model_cfg)
    names = select_module_names(model, ['backbone', 'encoder', 'decoder', 'query'], limit=18)
    print("Hooked modules:", *names, sep="\n- ")
    records = CocoDetectionRecords(paths.coco(DATASET_TRACK)/"val", paths.coco(DATASET_TRACK)/"annotations/instances_val.json")
    sample = records[0]["image"]
    outputs, handles = capture_module_outputs(model, names)
    prediction = adapter.predict([sample])[0]
    for handle in handles: handle.remove()
    display(draw_predictions(sample, prediction, run["class_names"], threshold=0.25))
    try:
        display(plot_activation_views(outputs))
    except RuntimeError as error:
        print(error)
else:
    print("Train or resume a completed run before executing this visualization cell.")


## Optional Optuna search

Trials use resumable SQLite storage in Drive and MedianPruner. The shared space covers LR, weight decay, warm-up, backbone LR multiplier, accumulation, resolution, augmentation, clipping, and max detections; model-specific spaces add RPN/RoI, hierarchy, or query/matcher parameters.

In [ ]:
if RUN_HYPERPARAMETER_SEARCH:
    !python scripts/tune.py --drive-root "$DRIVE_ROOT" --model-id $MODEL_ID --dataset-track $DATASET_TRACK --trials 20 --epochs-per-trial 20

## Final summary

The printed manifest contains best epoch, mAP, APtiny, total time, parameter counts, checkpoint paths, framework versions, GPU, and registered run ID.

In [ ]:
from src.training.checkpointing import RunRegistry
runs = RunRegistry(paths).list_available_runs(MODEL_ID, DATASET_TRACK, status=None)
runs[:3]